# Kuro Siwo — preprocess và kiểm thử
Upload **kuro_dataset.py** vào `/content` trên Colab, hoặc mở notebook từ repo local và điều chỉnh đường dẫn/mount phù hợp. Chạy lần lượt các cell. Pipeline nằm trong module Python; các hàm QC hiển thị trực tiếp bên dưới. Không cần các file Python phụ, requirements hay hướng dẫn riêng.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/kuro_siwo")
LOCAL_ROOT = Path("/content/kuro_siwo")
RAW_ON_DRIVE = DRIVE_ROOT / "raw" / "geobench_kuro_siwo.tortilla"
REPORT_ROOT = DRIVE_ROOT / "reports"
SCAN_ALL = True  # False: tối đa 3 mẫu/split, chỉ PASS_PARTIAL
BATCH_SIZE = 2
BENCHMARK_BATCHES = 50
NUM_WORKERS = 0
SAR_NOISE = False  # True cho thí nghiệm noise; baseline giữ False
NOISE_P = 0.3
NOISE_STD = 0.05

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *[
            "numpy>=1.26,<3",
            "pandas>=2.2,<4",
            "pyarrow>=17,<27",
            "rasterio>=1.4,<2",
            "matplotlib>=3.8,<4",
        ],
    ],
    check=True,
)
from google.colab import drive  # noqa: E402

drive.mount("/content/drive")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

## Import pipeline
Dùng torch có sẵn của Colab. Local cần cài torch tương thích. Không normalize thêm. Giữ official split và event ID; augmentation chỉ train.

In [ ]:
import importlib

CODE_DIR = next(
    (
        p
        for p in [Path("/content"), Path.cwd() / "scripts", Path.cwd().parent / "scripts"]
        if (p / "kuro_dataset.py").is_file()
    ),
    None,
)
if CODE_DIR is None:
    raise FileNotFoundError("Upload kuro_dataset.py vào /content trước.")
sys.path.insert(0, str(CODE_DIR))
import kuro_dataset  # noqa: E402

importlib.reload(kuro_dataset)
from kuro_dataset import RAW_NAME, KuroDataset, RawArchive, model_inputs, stage_raw  # noqa: E402

RAW_LOCAL = stage_raw(RAW_ON_DRIVE, LOCAL_ROOT / "raw" / RAW_NAME)
archive = RawArchive(RAW_LOCAL)
print("RAW:", RAW_LOCAL)
print(json.dumps(archive.audit, indent=2))

## 5. Đội model dùng Dataset và adapter
Đoạn dưới là ví dụ để dùng trong notebook train, không cần chạy thêm trước QC.
Normalization đã nằm trong loader; không normalize lần 2. Validation/test không
augmentation. `shuffle` và event-balanced sampler là trách nhiệm của trainer.

```python
from kuro_dataset import RawArchive, KuroDataset, model_inputs
from torch.utils.data import DataLoader
archive = RawArchive(RAW_LOCAL)  # verify một lần, dùng chung cho ba split
train_ds = KuroDataset(archive, 'train', augment=True)
val_ds = KuroDataset(archive, 'validation')
test_ds = KuroDataset(archive, 'test')
loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0)
batch = next(iter(loader))
resnet_input, terramind_input = model_inputs(batch)
```

ResNet nhận `[B,7,224,224]`; model team đổi first conv sang 7 kênh.
TerraMind **qua TemporalWrapper** nhận `S1GRD=[B,2,3,224,224]`,
`DEM=[B,1,3,224,224]`. Team model chốt checkpoint/pooling/decoder và chạy forward
thật. Mask int64 `[B,224,224]`, 0/1/2 và ignore 255. Batch toàn ignore phải bỏ
loss có log. Event/sample/AOI/split được giữ trong batch.



## Xem output preprocessing
SAR chuẩn hóa `[B,3,2,224,224]`, DEM `[B,1,224,224]`; mask int64 0/1/2 hoặc 255. Adapter CNN 7 kênh; TerraMind cần wrapper đa thời điểm. Pixel input không hợp lệ được điền 0 sau normalize. MixStyle/sampler thuộc đội model.

In [ ]:
from torch.utils.data import DataLoader

train_ds = KuroDataset(
    archive, "train", augment=True, sar_noise=SAR_NOISE, noise_p=NOISE_P, noise_std=NOISE_STD
)
val_ds = KuroDataset(archive, "validation")
test_ds = KuroDataset(archive, "test")
batch = next(iter(DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=0)))
cnn, temporal = model_inputs(batch)
print("CNN:", cnn.shape)
print("TerraMind:", {k: v.shape for k, v in temporal.items()})
print("Mask:", batch["mask"].shape, batch["mask"].dtype, batch["mask"].unique())
print("Event IDs:", batch["event_id"])

## Hàm QC, ảnh minh họa và benchmark
Chỉ dummy head kiểm tra loss/backward; chưa kiểm tra backbone thật. PASS_FULL yêu cầu checksum release đúng và scan toàn bộ. Benchmark CPU có thể dùng cache OS.

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from kuro_dataset import (
    CONFIG,
    EXPECTED_COUNTS,
    KuroDataset,
    RawArchive,
    model_inputs,
    now,
    transform_sample,
    validate_arrays,
    write_json,
)
from torch.utils.data import DataLoader

In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

In [ ]:
def figure(sample, path):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import BoundaryNorm, ListedColormap

    fig, axes = plt.subplots(1, 7, figsize=(21, 3))
    for i, title in enumerate(["pre1 VV (dB)", "pre2 VV (dB)", "event VV (dB)"]):
        axes[i].imshow(
            np.ma.masked_where(~sample["input_valid"], sample["sar_db"][i, 0]),
            cmap="gray",
            vmin=-25,
            vmax=0,
        )
        axes[i].set_title(title)
    axes[3].imshow(np.ma.masked_where(~sample["input_valid"], sample["dem_m"][0]), cmap="terrain")
    axes[3].set_title("DEM (m)")
    labels = np.where(sample["mask"] == 255, 3, sample["mask"])
    cmap = ListedColormap(["tan", "royalblue", "cyan", "black"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4)
    axes[4].imshow(labels, cmap=cmap, norm=norm)
    axes[4].set_title("0=land 1=water 2=flood 3=ignore")
    axes[5].imshow(sample["valid_mask"], vmin=0, vmax=1, cmap="gray")
    axes[5].set_title("valid supervision")
    axes[6].imshow(sample["sar_db"][2, 0], cmap="gray", vmin=-25, vmax=0)
    axes[6].imshow(
        np.ma.masked_where(~sample["valid_mask"], labels), cmap=cmap, norm=norm, alpha=0.4
    )
    axes[6].set_title("event + label overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(
        f"TRAIN sample {sample['sample_id']}, event {sample['event_id']}; display scaling only"
    )
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

In [ ]:
def benchmark(ds, batches=50, batch_size=2, workers=0):
    if batches < 1 or batch_size < 1 or workers < 0:
        raise ValueError("Invalid benchmark settings")
    if not len(ds):
        return {"status": "SKIPPED: empty train split"}
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers)
    iterator = iter(loader)
    start = time.perf_counter()
    next(iterator)
    first_batch_seconds = time.perf_counter() - start
    start = time.perf_counter()
    count, samples = 0, 0
    for _ in range(batches):
        try:
            batch = next(iterator)
        except StopIteration:
            break
        count += 1
        samples += len(batch["sample_id"])
    elapsed = time.perf_counter() - start
    return {
        "requested_batches": batches,
        "timed_batches": count,
        "timed_samples": samples,
        "first_batch_seconds": first_batch_seconds,
        "seconds": elapsed,
        "samples_per_second": samples / elapsed if count else None,
        "num_workers": workers,
        "batch_size": batch_size,
        "scope": "CPU data loading after QC; OS cache may be warm; not GPU training speed",
    }

In [ ]:
def run(
    raw, report_dir, *, scan_all=True, batch_size=2, benchmark_batches=50, workers=0, archive=None
):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    archive = archive if archive is not None else RawArchive(raw)
    archive.save_metadata(report_dir)
    print("OFFICIAL SPLIT AUDIT:", json.dumps(archive.audit, indent=2), flush=True)
    examples, qc = {}, {}
    preview = None
    for split in EXPECTED_COUNTS:
        ds = KuroDataset(archive, split, augment=False)
        checked, all_ignore, labels = 0, [], Counter()
        count = len(ds) if scan_all else min(3, len(ds))
        for i in range(count):
            sample = ds.physical(i)
            validate_arrays(sample)
            labels.update({str(c): int((sample["mask"] == c).sum()) for c in [0, 1, 2, 255]})
            if not sample["valid_mask"].any():
                all_ignore.append(sample["sample_id"])
            if split == "train" and (
                preview is None
                or (not (preview["mask"] == 2).any() and (sample["mask"] == 2).any())
            ):
                preview = sample  # visualization selection uses train labels only
            checked += 1
            if checked % 250 == 0:
                print(f"QC {split}: {checked}/{count}", flush=True)
        qc[split] = {
            "total_samples": len(ds),
            "checked_samples": checked,
            "all_ignore_ids": all_ignore,
            "class_pixels": dict(labels),
        }
        if not len(ds):
            continue
        batch = next(iter(DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)))
        cnn, tm = model_inputs(batch)
        b = len(batch["sample_id"])
        require(tuple(cnn.shape) == (b, 7, 224, 224), "CNN input shape")
        require(tuple(tm["S1GRD"].shape) == (b, 2, 3, 224, 224), "TM SAR shape")
        require(tuple(tm["DEM"].shape) == (b, 1, 3, 224, 224), "TM DEM shape")
        require(torch.equal(cnn[:, :6], batch["sar"].flatten(1, 2)), "Wrong channel order")
        require(torch.equal(tm["S1GRD"].permute(0, 2, 1, 3, 4), batch["sar"]), "Wrong time order")
        require(batch["mask"].dtype == torch.int64, "Wrong mask dtype")
        require(all(torch.isfinite(x).all().item() for x in [cnn, *tm.values()]), "Nonfinite input")
        invalid = ~batch["input_valid"]
        require(
            torch.all(cnn.masked_select(invalid[:, None].expand_as(cnn)) == 0).item(),
            "Invalid fill",
        )
        loss_status = "SKIPPED: all-ignore batch"
        if (batch["mask"] != 255).any():
            torch.manual_seed(42)
            head = torch.nn.Conv2d(7, 3, 1)
            loss = torch.nn.functional.cross_entropy(head(cnn), batch["mask"], ignore_index=255)
            loss.backward()
            require(torch.isfinite(loss).item(), "Nonfinite dummy loss")
            require(
                all(torch.isfinite(p.grad).all().item() for p in head.parameters()), "Bad gradient"
            )
            loss_status = f"PASS (dummy head only), loss={loss.item():.6f}"
        examples[split] = {
            "resnet50_input": list(cnn.shape),
            "terramind_inputs": {k: list(v.shape) for k, v in tm.items()},
            "mask_shape": list(batch["mask"].shape),
            "mask_dtype": str(batch["mask"].dtype),
            "labels_in_batch": torch.unique(batch["mask"]).tolist(),
            "sample_ids": batch["sample_id"],
            "event_ids": batch["event_id"],
            "dummy_loss_backward": loss_status,
        }
        print(json.dumps({split: examples[split]}, indent=2), flush=True)
    if preview is not None:
        figure(preview, report_dir / "train_preview.png")
        figure(transform_sample(preview, d4=1), report_dir / "train_rot90_preview.png")
    speed = benchmark(KuroDataset(archive, "train"), benchmark_batches, batch_size, workers)
    ready = archive.verified_release and scan_all
    result = {
        "status": "PASS_FULL" if ready else "PASS_PARTIAL",
        "checked_at": now(),
        "verified_release": archive.verified_release,
        "full_raster_scan": scan_all,
        "config": CONFIG,
        "audit": archive.audit,
        "qc": qc,
        "examples": examples,
        "benchmark": speed,
        "real_resnet50_forward": "NOT RUN",
        "real_terramind_forward": "NOT RUN",
        "DG_validation": "NOT RUN",
    }
    write_json(report_dir / "test_report.json", result)
    print("BENCHMARK:", json.dumps(speed, indent=2), flush=True)
    print(f"{result['status']}: raw + data adapters checked. Actual models NOT RUN.", flush=True)
    print("REPORT_DIR:", report_dir, flush=True)
    return result

## SAR noise tùy chọn — kiểm thử bật/tắt
`SAR_NOISE=False` giữ baseline; `True` bật augmentation Gamma với xác suất 0.3/mẫu,
độ lệch chuẩn hệ số nhân 0.05 (không phải giới hạn ±5%). Đây là cấu hình thử,
không phải số looks của cảm biến hay mức tối ưu đã kiểm chứng. Nhân trên linear
power trước dB; độc lập theo pixel/kênh/thời điểm, chỉ train và input_valid.
D4 vẫn dùng `augment=True` độc lập. Không thêm noise vào DEM/mask/val/test.

Đội train tạo hai run riêng; giữ split, seed và cấu hình khác giống nhau, ghi
`train_ds.augmentation_config` vào run record. Thêm noise tiêu thụ RNG nên không
đảm bảo chuỗi D4 về sau trùng giữa hai run. Nếu cần, trainer dùng generator riêng
cho sampler để giữ thứ tự mẫu. Chọn bằng validation, không tune theo test.

Cell dưới kiểm tra cả hai đường code dù SAR_NOISE đang tắt. QC ép p=1 để chắc
chắn kiểm tra nhánh noise; đây không phải cấu hình thí nghiệm mặc định.
`physical(i)` luôn trả mẫu không noise; `physical(i, apply_noise=True)` mới áp dụng
cấu hình noise. `ds[i]` tự áp dụng cấu hình. Không có bản processed mới được lưu.


In [ ]:
def check_sar_noise(archive):
    import torch
    from kuro_dataset import multiplicative_sar_noise, transform_sample

    # fork_rng keeps QC from advancing the training CPU RNG.
    with torch.random.fork_rng(devices=[]):
        base = KuroDataset(archive, "train")
        off = KuroDataset(archive, "train", sar_noise=False)
        zero = KuroDataset(archive, "train", sar_noise=True, noise_p=0)
        on = KuroDataset(archive, "train", sar_noise=True, noise_p=1, noise_std=0.05)
        index = next(
            (i for i in range(min(len(base), 100)) if base.physical(i)["input_valid"].any()), None
        )
        if index is None:
            raise ValueError("No valid train sample in first 100; cannot check noise branch")
        clean = base.physical(index)
        a = base[index]
        for ds in (off, zero):
            state = torch.get_rng_state().clone()
            b = ds[index]
            assert torch.equal(state, torch.get_rng_state()), "Disabled noise consumed RNG"
            for key in a:
                assert (
                    torch.equal(a[key], b[key]) if torch.is_tensor(a[key]) else a[key] == b[key]
                ), key
        torch.manual_seed(42)
        noisy = on.physical(index, apply_noise=True)
        torch.manual_seed(42)
        repeat = on.physical(index, apply_noise=True)
        np.testing.assert_array_equal(noisy["sar_db"], repeat["sar_db"])
        assert not np.array_equal(clean["sar_db"], noisy["sar_db"])
        validate_arrays(noisy)
        for key in ("dem_m", "mask", "input_valid", "valid_mask"):
            np.testing.assert_array_equal(clean[key], noisy[key])
        for key in ("sample_id", "event_id", "aoi_id", "split"):
            assert clean[key] == noisy[key]
        norm = CONFIG["normalization"]
        expected = (noisy["sar_db"] - np.array(norm["sar_mean"])[None, :, None, None]) / np.array(
            norm["sar_std"]
        )[None, :, None, None]
        expected[..., ~noisy["input_valid"]] = 0
        for d4 in range(8):
            transformed = transform_sample(noisy, d4)
            clean_transformed = transform_sample(clean, d4)
            for key in ("dem", "mask", "input_valid", "valid_mask"):
                np.testing.assert_array_equal(transformed[key], clean_transformed[key])
        np.testing.assert_allclose(transform_sample(noisy)["sar"], expected, atol=2e-6)
        torch.manual_seed(42)
        item = on[index]
        np.testing.assert_allclose(item["sar"].numpy(), transform_sample(noisy)["sar"])
        # Boundary probability and validation/test guards.
        for split in ("validation", "test"):
            try:
                KuroDataset(archive, split, sar_noise=True)
            except ValueError:
                pass
            else:
                raise AssertionError("Noise accepted outside train")
        for kwargs in (
            {"noise_p": -0.1},
            {"noise_p": 1.1},
            {"noise_p": float("nan")},
            {"noise_std": 0},
            {"noise_std": float("inf")},
            {"noise_std": 2},
        ):
            try:
                KuroDataset(archive, "train", **kwargs)
            except ValueError:
                pass
            else:
                raise AssertionError(f"Invalid parameters accepted: {kwargs}")
        # Check masks, no mutation, all-invalid case, multiplier moments and independence.
        power = np.ones((3, 2, 224, 224), np.float32)
        valid = np.ones((224, 224), bool)
        valid[0] = False
        power[..., 0, :] = 0
        original = power.copy()
        torch.manual_seed(123)
        changed = multiplicative_sar_noise(power, valid, 0.05)
        np.testing.assert_array_equal(power, original)
        np.testing.assert_array_equal(changed[..., ~valid], original[..., ~valid])
        np.testing.assert_array_equal(
            multiplicative_sar_noise(power, np.zeros_like(valid), 0.05), original
        )
        factors = changed[..., valid]
        assert np.isfinite(factors).all() and (factors > 0).all()
        assert abs(float(factors.mean()) - 1) < 0.001
        assert abs(float(factors.std()) - 0.05) < 0.001
        assert not np.array_equal(changed[0, 0], changed[0, 1])
        assert not np.array_equal(changed[0, 0], changed[1, 0])
        # p=0.3 actually takes both branches; deterministic seed for this smoke check.
        random_ds = KuroDataset(archive, "train", sar_noise=True, noise_p=0.3)
        torch.manual_seed(123)
        decisions = [
            not np.array_equal(
                clean["sar_db"], random_ds.physical(index, apply_noise=True)["sar_db"]
            )
            for _ in range(20)
        ]
        assert any(decisions) and not all(decisions)
    return (
        {
            "status": "PASS",
            "forced_noise_config": on.augmentation_config,
            "multiplier_mean": float(factors.mean()),
            "multiplier_std": float(factors.std()),
            "real_model_training": "NOT RUN",
        },
        clean,
        noisy,
    )

## Chạy test và in kết quả
Report JSON, metadata/config, environment và ảnh train lưu trong REPORT_DIR trên Drive. Không xuất thêm bản processed. Batch toàn ignore cần trainer bỏ loss và ghi nhận.

In [ ]:
from datetime import UTC, datetime

from IPython.display import Image, display

REPORT_DIR = REPORT_ROOT / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
(REPORT_DIR / "pipeline_sha256.txt").write_text(kuro_dataset.sha256(Path(kuro_dataset.__file__)))
try:
    report = run(
        RAW_LOCAL,
        REPORT_DIR,
        scan_all=SCAN_ALL,
        batch_size=BATCH_SIZE,
        benchmark_batches=BENCHMARK_BATCHES,
        workers=NUM_WORKERS,
        archive=archive,
    )
except Exception as error:
    write_json(
        REPORT_DIR / "test_report.json", {"status": "FAIL", "at": now(), "error": str(error)}
    )
    raise
print("STATUS:", report["status"])
print("REPORT:", REPORT_DIR)
for name in ["train_preview.png", "train_rot90_preview.png"]:
    if (REPORT_DIR / name).exists():
        display(Image(filename=str(REPORT_DIR / name)))
try:
    noise_report, clean_sample, noisy_sample = check_sar_noise(archive)
except Exception as error:
    write_json(REPORT_DIR / "sar_noise_test.json", {"status": "FAIL", "error": str(error)})
    raise
write_json(REPORT_DIR / "sar_noise_test.json", noise_report)
write_json(REPORT_DIR / "train_augmentation_config.json", train_ds.augmentation_config)
print("SAR NOISE QC:", noise_report)
print("SELECTED TRAIN CONFIG:", train_ds.augmentation_config)

import matplotlib.pyplot as plt  # noqa: E402

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
valid = clean_sample["input_valid"]
for ax, sample, title in zip(
    axes[:2],
    (clean_sample, noisy_sample),
    ("Train post VV: baseline", "Train post VV: forced noise"),
    strict=True,
):
    ax.imshow(np.ma.masked_where(~valid, sample["sar_db"][2, 0]), cmap="gray", vmin=-25, vmax=0)
    ax.set_title(title)
delta = noisy_sample["sar_db"][2, 0] - clean_sample["sar_db"][2, 0]
im = axes[2].imshow(np.ma.masked_where(~valid, delta), cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_title("Noise minus baseline (dB)")
fig.colorbar(im, ax=axes[2])
fig.tight_layout()
fig.savefig(REPORT_DIR / "sar_noise_comparison.png", dpi=120)
plt.show()
plt.close(fig)